In [1]:
import pandas as pd
from scipy.stats import randint, uniform
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer

Загружаем датасет и создаём датафрейм:

In [2]:
df = pd.read_csv('../data/student_performance_finalscore.csv')

In [3]:
df

,Student_ID,Age,Gender,Hours_Studied,Attendance,Sleep_Hours,Stress_Level,Screen_Time,Previous_GPA,Part_Time_Job,Study_Method,Diet_Quality,Internet_Quality,Extracurricular,Tutoring_Sessions_Per_Week,Family_Income_Level,Exam_Anxiety_Score,Final_Score
0,STU00001,17,Female,8.20,80.0,7.3,2.3,3.2,3.07,No,Online,Average,Average,Yes,3,High,1.8,87.10
1,STU00002,22,Male,4.07,69.2,7.9,3.6,0.5,2.66,Yes,Offline,Average,Good,No,2,Middle,1.0,92.03
2,STU00003,23,Female,5.07,74.2,6.7,2.6,4.7,3.27,No,Hybrid,Average,Good,No,1,Middle,4.2,95.37
3,STU00004,22,Female,5.82,82.5,3.7,3.3,2.3,2.87,No,Offline,Average,Excellent,No,0,Middle,3.6,90.73
4,STU00005,21,Male,3.42,90.8,7.4,7.0,3.1,2.82,Yes,Offline,Average,Good,No,2,Middle,4.5,74.71
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7995,STU07996,18,Female,6.45,93.6,9.7,5.4,5.9,3.28,No,Online,Poor,Excellent,Yes,3,Middle,3.8,98.58
7996,STU07997,20,Non-Binary,0.80,94.4,7.6,6.7,2.2,3.73,Yes,Offline,Poor,Good,Yes,2,Middle,4.5,72.58
7997,STU07998,19,Female,3.76,96.4,6.8,6.9,3.1,1.81,No,Hybrid,Poor,Good,No,0,Low,2.2,63.95
7998,STU07999,20,Female,3.87,79.2,8.2,8.2,4.9,3.36,No,Hybrid,Good,Excellent,Yes,2,Low,3.1,85.19


Проверим количество пропусков в датафрейме:

In [4]:
df.isna().sum()

Student_ID                    0
Age                           0
Gender                        0
Hours_Studied                 0
Attendance                    0
Sleep_Hours                   0
Stress_Level                  0
Screen_Time                   0
Previous_GPA                  0
Part_Time_Job                 0
Study_Method                  0
Diet_Quality                  0
Internet_Quality              0
Extracurricular               0
Tutoring_Sessions_Per_Week    0
Family_Income_Level           0
Exam_Anxiety_Score            0
Final_Score                   0
dtype: int64

Рассмотрим статистические показатели для признаков датасета:

In [5]:
df.describe()

,Age,Hours_Studied,Attendance,Sleep_Hours,Stress_Level,Screen_Time,Previous_GPA,Tutoring_Sessions_Per_Week,Exam_Anxiety_Score,Final_Score
count,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000
mean,20.494375,4.983845,79.933375,6.989125,5.014175,4.024525,2.992408,1.700625,4.494238,83.205649
std,2.285962,1.951715,9.656594,1.192898,1.940126,1.481908,0.489530,1.112836,1.685571,12.756728
min,17.000000,0.020000,43.300000,3.000000,1.000000,0.500000,1.500000,0.000000,1.000000,22.810000
25%,18.000000,3.680000,73.400000,6.200000,3.700000,3.000000,2.670000,1.000000,3.300000,75.507500
50%,20.500000,4.980000,80.100000,7.000000,5.000000,4.000000,2.990000,2.000000,4.400000,86.510000
75%,22.000000,6.312500,86.600000,7.800000,6.300000,5.000000,3.330000,2.000000,5.600000,93.080000
max,24.000000,12.000000,100.000000,10.000000,10.000000,9.600000,6.700000,5.000000,10.000000,99.980000


Выполним предобработку данных: 
1) отфильтруем наблюдения по условию (уберём записи, в которых Gender = 'Non-Binary');
2) удалим наименее содержательные признаки (ID студента, Внеклассную деятельность, Ежедневный рацион питания и метку);
3) разделим данные на признаки (X) и целевую переменную (y).

In [6]:
df.head(15)

,Student_ID,Age,Gender,Hours_Studied,Attendance,Sleep_Hours,Stress_Level,Screen_Time,Previous_GPA,Part_Time_Job,Study_Method,Diet_Quality,Internet_Quality,Extracurricular,Tutoring_Sessions_Per_Week,Family_Income_Level,Exam_Anxiety_Score,Final_Score
0,STU00001,17,Female,8.20,80.0,7.3,2.3,3.2,3.07,No,Online,Average,Average,Yes,3,High,1.8,87.10
1,STU00002,22,Male,4.07,69.2,7.9,3.6,0.5,2.66,Yes,Offline,Average,Good,No,2,Middle,1.0,92.03
2,STU00003,23,Female,5.07,74.2,6.7,2.6,4.7,3.27,No,Hybrid,Average,Good,No,1,Middle,4.2,95.37
3,STU00004,22,Female,5.82,82.5,3.7,3.3,2.3,2.87,No,Offline,Average,Excellent,No,0,Middle,3.6,90.73
4,STU00005,21,Male,3.42,90.8,7.4,7.0,3.1,2.82,Yes,Offline,Average,Good,No,2,Middle,4.5,74.71
5,STU00006,22,Female,5.00,82.7,6.5,9.1,3.8,3.16,No,Hybrid,Good,Average,No,1,High,1.4,93.98
6,STU00007,17,Male,5.00,78.9,5.0,7.8,3.0,2.92,No,Online,Average,Good,Yes,1,Middle,4.2,64.99
7,STU00008,19,Male,1.49,77.4,6.0,6.2,4.7,2.08,No,Offline,Average,Average,Yes,0,Middle,7.1,44.64
8,STU00009,21,Female,7.04,95.6,6.7,3.0,2.7,3.28,No,Online,Poor,Poor,No,2,High,4.0,96.95
9,STU00010,19,Female,6.20,71.9,6.4,4.6,5.0,2.05,No,Hybrid,Average,Average,Yes,0,High,4.3,81.24


In [7]:
filtered_df = df[df['Gender'] != 'Non-Binary']

In [8]:
X = filtered_df.drop(columns=['Student_ID', 'Extracurricular', 'Diet_Quality', 'Final_Score'])

In [9]:
X

,Age,Gender,Hours_Studied,Attendance,Sleep_Hours,Stress_Level,Screen_Time,Previous_GPA,Part_Time_Job,Study_Method,Internet_Quality,Tutoring_Sessions_Per_Week,Family_Income_Level,Exam_Anxiety_Score
0,17,Female,8.20,80.0,7.3,2.3,3.2,3.07,No,Online,Average,3,High,1.8
1,22,Male,4.07,69.2,7.9,3.6,0.5,2.66,Yes,Offline,Good,2,Middle,1.0
2,23,Female,5.07,74.2,6.7,2.6,4.7,3.27,No,Hybrid,Good,1,Middle,4.2
3,22,Female,5.82,82.5,3.7,3.3,2.3,2.87,No,Offline,Excellent,0,Middle,3.6
4,21,Male,3.42,90.8,7.4,7.0,3.1,2.82,Yes,Offline,Good,2,Middle,4.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7994,24,Female,1.09,76.6,8.5,2.3,3.5,4.00,No,Offline,Excellent,2,Middle,3.1
7995,18,Female,6.45,93.6,9.7,5.4,5.9,3.28,No,Online,Excellent,3,Middle,3.8
7997,19,Female,3.76,96.4,6.8,6.9,3.1,1.81,No,Hybrid,Good,0,Low,2.2
7998,20,Female,3.87,79.2,8.2,8.2,4.9,3.36,No,Hybrid,Excellent,2,Low,3.1


In [10]:
y = filtered_df['Final_Score']

In [11]:
X.shape

(7716, 14)

In [12]:
y.shape

(7716,)

In [13]:
X_train, X_test, y_train, y_test = train_test_split (X, y, test_size = 0.3, random_state = 54)

Рассмотрим получившиеся размерности обучающей и тестовой выборок:

In [14]:
X_train.shape

(5401, 14)

In [15]:
X_test.shape

(2315, 14)

Перед написанием конвейера обработки данных (Pipeline) необходимо разделить признаки на категориальные и числовые:

1) В рассматриваемом датасете категориальные признаки делятся на номинальные (Gender, Study_Method), порядковые (Internet_Quality, Family_Income_Level) и бинарные (Part_Time_Job). Для порядковых признаков применяется OrdinalEncoder, поскольку между их значениями существует естественный порядок. Для номинальных и бинарных признаков используется OneHotEncoder, так как между категориями отсутствует упорядоченность.
2) Для модели RandomForestRegressor масштабирование числовых признаков не применяется, поскольку деревья решений и ансамбли на их основе не чувствительны к масштабу данных.

In [16]:
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, RandomizedSearchCV, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

Определим количество уникальных значений в порядковых признаках Internet_Quality и Family_Income_Level.

In [17]:
df['Internet_Quality'].unique()

array(['Average', 'Good', 'Excellent', 'Poor'], dtype=object)

In [18]:
df['Family_Income_Level'].unique()

array(['High', 'Middle', 'Low'], dtype=object)

Приступим к созданию конвейеров:

In [19]:
ohe_features = ['Gender', 'Study_Method', 'Part_Time_Job']

ohe_transformer = Pipeline(
    steps = [
        ('impute', SimpleImputer(strategy = 'most_frequent')),
        ('one-hot-encoder', OneHotEncoder(handle_unknown = 'ignore', sparse_output = False, drop='first')),
    ]
)

In [20]:
ord_features = ['Internet_Quality', 'Family_Income_Level']

ord_transformer = Pipeline(
    steps = [
        ('impute', SimpleImputer(strategy = 'most_frequent')),
        ('ordinal-encoder', OrdinalEncoder(categories = [
            ['Poor', 'Average', 'Good', 'Excellent'],
            ['Low', 'Middle', 'High']], 
            handle_unknown = 'use_encoded_value',
            unknown_value = -1)),
    ]
)

In [21]:
num_features = ['Age', 'Hours_Studied', 'Attendance', 'Sleep_Hours', 'Stress_Level', 'Screen_Time', 'Previous_GPA', 'Tutoring_Sessions_Per_Week', 'Exam_Anxiety_Score']

num_transformer = Pipeline(
    steps = [
        ('impute', SimpleImputer(strategy = 'mean')),
        #('scale', StandardScaler()),
    ]
)

Объединяем наши конвейеры:

In [22]:
preprocessor = ColumnTransformer(
    transformers = [
        ('numeric', num_transformer, num_features),
        ('cat_ordinal',ord_transformer, ord_features),
        ('cat_onehot', ohe_transformer, ohe_features),
    ],
    remainder = 'drop',
    n_jobs = -1 #чтобы задействовать все ядра CPU
)

Рассматриваемые модели:

In [23]:
rf = RandomForestRegressor()

Итоговый конвейер:

In [24]:
pipefinal = Pipeline([
    ('preprocessor', preprocessor),
    ('model', rf)
])

Настройка гиперпараметров:

In [25]:
kf = KFold(n_splits = 15, shuffle = True, random_state = 13)

In [26]:
param_dist = {
    'model__n_estimators': [100, 200, 300, 500],
    'model__max_depth': [8, 10, 15],
    'model__min_samples_split': [5, 10, 15],
    'model__min_samples_leaf': [2, 4, 6]
}

In [27]:
scoring = {
    'MAE': 'neg_mean_absolute_error',
    'MSE': 'neg_mean_squared_error',
    'R2': 'r2'
}

In [28]:
random_cv = RandomizedSearchCV(
    estimator=pipefinal,
    param_distributions=param_dist,
    n_iter=20,
    cv=kf,
    scoring=scoring,
    refit='MSE',
    random_state=42,
    n_jobs=-1
)

In [29]:
random_cv.fit(X_train, y_train)

RandomizedSearchCV(cv=KFold(n_splits=15, random_state=13, shuffle=True),
                   estimator=Pipeline(steps=[('preprocessor',
                                              ColumnTransformer(n_jobs=-1,
                                                                transformers=[('numeric',
                                                                               Pipeline(steps=[('impute',
                                                                                                SimpleImputer())]),
                                                                               ['Age',
                                                                                'Hours_Studied',
                                                                                'Attendance',
                                                                                'Sleep_Hours',
                                                                                'Stress_Level',
                                                                                'Screen_Time',
                                                                                'Previous_GPA',
                                                                                'Tutoring_Sessions_Per_Week',
                                                                                'Exam_Anxiety_Sco...
                                                                                'Part_Time_Job'])])),
                                             ('model',
                                              RandomForestRegressor())]),
                   n_iter=20, n_jobs=-1,
                   param_distributions={'model__max_depth': [8, 10, 15],
                                        'model__min_samples_leaf': [2, 4, 6],
                                        'model__min_samples_split': [5, 10, 15],
                                        'model__n_estimators': [100, 200, 300,
                                                                500]},
                   random_state=42, refit='MSE',
                   scoring={'MAE': 'neg_mean_absolute_error',
                            'MSE': 'neg_mean_squared_error', 'R2': 'r2'})

In [30]:
random_cv.best_params_

{'model__n_estimators': 500,
 'model__min_samples_split': 10,
 'model__min_samples_leaf': 2,
 'model__max_depth': 15}

In [31]:
random_cv.best_estimator_

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(n_jobs=-1,
                                   transformers=[('numeric',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer())]),
                                                  ['Age', 'Hours_Studied',
                                                   'Attendance', 'Sleep_Hours',
                                                   'Stress_Level',
                                                   'Screen_Time',
                                                   'Previous_GPA',
                                                   'Tutoring_Sessions_Per_Week',
                                                   'Exam_Anxiety_Score']),
                                                 ('cat_ordinal',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(strategy='most_freq...
                                                  ['Internet_Quality',
                                                   'Family_Income_Level']),
                                                 ('cat_onehot',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('one-hot-encoder',
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['Gender', 'Study_Method',
                                                   'Part_Time_Job'])])),
                ('model',
                 RandomForestRegressor(max_depth=15, min_samples_leaf=2,
                                       min_samples_split=10,
                                       n_estimators=500))])

In [32]:
model = random_cv.best_estimator_

In [33]:
y_pred = model.predict(X_test)

In [34]:
MSE = mean_squared_error(y_test, y_pred)
MAE = mean_absolute_error(y_test, y_pred)

In [35]:
print(f"MSE - {MSE}")
print(f"MAE - {MAE}")

MSE - 48.92405034339636
MAE - 5.481078521776955


In [36]:
model.score(X_test, y_test)

0.7027277388807026

In [37]:
model.score(X_train, y_train)

0.8947467578993171

Сохраняем модель:

In [38]:
import joblib

In [39]:
joblib.dump (model, "../models/model_pipeline.pkl")

['../models/model_pipeline.pkl']